# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Teninlanimi/flyrank-interenship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

 One row: One content page for one client on one reporting day.
 One row = one report_date + one client_hash_id + one content_hash_id in fact_content_daily_performance.

Dates covered in the warehouse release span 2025-01-27 through 2026-06-30.

In [15]:
import os
from huggingface_hub import HfApi, hf_hub_download
import pyarrow.parquet as pq

repo_id = "FlyRank/internship-warehouse"
token = os.environ["HF_TOKEN"]
api = HfApi()

files = api.list_repo_files(repo_id, repo_type="dataset", token=token)
months = sorted({f.split("/")[1].split("=")[1] for f in files if f.startswith("fact_content_daily_performance/month=")})
print("Available months:", months[0], "to", months[-1])

for filename in [
    "fact_content_daily_performance/month=2025-01/data_0.parquet",
    "fact_content_daily_performance/month=2026-06/data_0.parquet",
]:
    path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=filename, token=token)
    df = pq.read_table(path, columns=["report_date", "client_hash_id", "content_hash_id"]).to_pandas()
    print(filename)
    print("  report_date range:", df["report_date"].min(), "to", df["report_date"].max())
    print("  rows:", len(df))
    print("  duplicate grain rows:",
          df.groupby(["report_date", "client_hash_id", "content_hash_id"]).size().gt(1).sum())

Available months: 2025-01 to 2026-06
fact_content_daily_performance/month=2025-01/data_0.parquet
  report_date range: 2025-01-27 to 2025-01-31
  rows: 1297
  duplicate grain rows: 0
fact_content_daily_performance/month=2026-06/data_0.parquet
  report_date range: 2026-06-01 to 2026-06-30
  rows: 11694072
  duplicate grain rows: 6390


In [16]:
%pip install huggingface_hub pyarrow pandas

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature

These are the variables I plan to use as model inputs because they are available before making a refresh decision.

impressions
clicks
ctr
avg_position
content_age_days

These features describe a page's historical search performance and age at the decision point.

Label

The warehouse does not contain a ready-made target variable.

For Lane 2, the prediction target must be derived from future performance (for example, future traffic trend or refresh opportunity). Therefore, no raw warehouse column is used directly as the label.

Label: Derived refresh-opportunity target (created from future performance window).

Context

These fields provide metadata, time information, or are used for joins but are not model features.

report_date
content_type
content_created_date
content_updated_date
client_hash_id
content_hash_id

These fields define when the observation occurred and identify the content but are not directly used as predictive inputs.

Excluded
client_hash_id-	Identifier only; no predictive meaning
content_hash_id	-Identifier only
keyword_hash_id - oin key, not a feature
url_hash_id	- join key
query_hash_id- Join key
is_deleted - Deleted content is not eligible for refresh ranking
Future performance columns - Would leak information unavailable at the decision time.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:

import os
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq
import pandas as pd

token = os.environ["HF_TOKEN"]
repo_id = "FlyRank/internship-warehouse"

def download_and_read(filename, columns=None):
    """Helper to download and read a parquet file."""
    path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=filename, token=token)
    return pq.read_table(path, columns=columns).to_pandas()

# ==================== GRAIN VERIFICATION ====================
print("=== GRAIN: One row = client + content + report_date ===\n")

# Download a sample month
df_fact = download_and_read(
    "fact_content_daily_performance/month=2025-01/data_0.parquet",
    columns=["report_date", "client_hash_id", "content_hash_id"]
)

# Check grain: count rows vs unique combinations
num_rows = len(df_fact)
grain_check = df_fact.groupby(["report_date", "client_hash_id", "content_hash_id"]).size()
duplicates = (grain_check > 1).sum()

print(f"Total rows: {num_rows}")
print(f"Unique (report_date, client_hash_id, content_hash_id): {len(grain_check)}")
print(f"Duplicate grain combinations: {duplicates}")
print(f"Grain verified: {duplicates == 0}\n")

# ==================== ROW COUNTS ====================
print("=== ROW COUNTS ===\n")

dim_clients = download_and_read("dim_clients.parquet", columns=["client_hash_id"])
dim_content = download_and_read("dim_content.parquet", columns=["content_hash_id"])
fact_query_90d = download_and_read("fact_content_query_90d.parquet", columns=["client_hash_id"])

print(f"dim_clients: {len(dim_clients)} rows")
print(f"dim_content: {len(dim_content)} rows")
print(f"fact_content_daily_performance (2025-01 sample): {len(df_fact)} rows")
print(f"fact_content_query_90d: {len(fact_query_90d)} rows\n")

# ==================== DATE WINDOWS ====================
print("=== DATE WINDOWS ===\n")

# Check all months available
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files(repo_id, repo_type="dataset", token=token)
fact_files = sorted(p for p in files if p.startswith("fact_content_daily_performance/month="))
months = sorted({p.split("/")[1].split("=")[1] for p in fact_files if "month=" in p})

print(f"Available months: {months[0]} to {months[-1]}")
print(f"Total months: {len(months)}\n")

# Download first and last months to verify date coverage
first_month_data = download_and_read("fact_content_daily_performance/month=2025-01/data_0.parquet", columns=["report_date"])
last_month_data = download_and_read("fact_content_daily_performance/month=2026-06/data_0.parquet", columns=["report_date"])

print(f"First month (2025-01):")
print(f"  Date range: {first_month_data['report_date'].min()} to {first_month_data['report_date'].max()}")
print(f"  Rows: {len(first_month_data)}\n")

print(f"Last month (2026-06):")
print(f"  Date range: {last_month_data['report_date'].min()} to {last_month_data['report_date'].max()}")
print(f"  Rows: {len(last_month_data)}\n")

# ==================== MISSING VALUES ====================
print("=== MISSING VALUES (Features) ===\n")

df_fact_full = download_and_read(
    "fact_content_daily_performance/month=2025-01/data_0.parquet",
    columns=["gsc_impressions", "gsc_clicks", "ga4_pageviews", "ga4_sessions", "scroll_events"]
)

print("fact_content_daily_performance (2025-01 sample):")
for col in df_fact_full.columns:
    null_count = df_fact_full[col].isna().sum()
    null_pct = (null_count / len(df_fact_full)) * 100
    print(f"  {col}: {null_count} nulls ({null_pct:.1f}%)")

print()

dim_content_full = download_and_read(
    "dim_content.parquet",
    columns=["search_volume", "competition", "cpc", "backlinks", "keyword_char_count"]
)

print("dim_content:")
for col in dim_content_full.columns:
    null_count = dim_content_full[col].isna().sum()
    null_pct = (null_count / len(dim_content_full)) * 100
    print(f"  {col}: {null_count} nulls ({null_pct:.1f}%)")

print()

# ==================== FOREIGN KEY VERIFICATION ====================
print("=== FOREIGN KEY CHECKS ===\n")

df_fact_keys = download_and_read(
    "fact_content_daily_performance/month=2025-01/data_0.parquet",
    columns=["client_hash_id", "content_hash_id"]
)

clients_in_fact = set(df_fact_keys["client_hash_id"].unique())
clients_in_dim = set(dim_clients["client_hash_id"].unique())

contents_in_fact = set(df_fact_keys["content_hash_id"].unique())
contents_in_dim = set(dim_content["content_hash_id"].unique())

print(f"Fact table client_hash_id values exist in dim_clients: {clients_in_fact.issubset(clients_in_dim)}")
print(f"  Clients in fact: {len(clients_in_fact)}, in dim: {len(clients_in_dim)}")

print(f"Fact table content_hash_id values exist in dim_content: {contents_in_fact.issubset(contents_in_dim)}")
print(f"  Contents in fact: {len(contents_in_fact)}, in dim: {len(contents_in_dim)}")

print()

# ==================== QUERY 90D WINDOW ====================
print("=== fact_content_query_90d WINDOWS ===\n")

fact_query = download_and_read(
    "fact_content_query_90d.parquet",
    columns=["window_start", "window_end"]
)

print(f"Query 90d window_start range: {fact_query['window_start'].min()} to {fact_query['window_start'].max()}")
print(f"Query 90d window_end range: {fact_query['window_end'].min()} to {fact_query['window_end'].max()}")
print(f"Rows: {len(fact_query)}\n")

# ==================== FIELD VALUE RANGES ====================
print("=== FEATURE VALUE RANGES ===\n")

df_ranges = download_and_read(
    "fact_content_daily_performance/month=2025-01/data_0.parquet",
    columns=["gsc_avg_position", "ga4_pageviews", "scroll_events"]
)

print("fact_content_daily_performance:")
print(f"  gsc_avg_position: min={df_ranges['gsc_avg_position'].min()}, max={df_ranges['gsc_avg_position'].max()}, mean={df_ranges['gsc_avg_position'].mean():.2f}")
print(f"  ga4_pageviews: min={df_ranges['ga4_pageviews'].min()}, max={df_ranges['ga4_pageviews'].max()}, mean={df_ranges['ga4_pageviews'].mean():.2f}")
print(f"  scroll_events: min={df_ranges['scroll_events'].min()}, max={df_ranges['scroll_events'].max()}, mean={df_ranges['scroll_events'].mean():.2f}")

print("\nAll verification queries complete ✓")

=== GRAIN: One row = client + content + report_date ===

Total rows: 1297
Unique (report_date, client_hash_id, content_hash_id): 1297
Duplicate grain combinations: 0
Grain verified: True

=== ROW COUNTS ===

dim_clients: 104 rows
dim_content: 519606 rows
fact_content_daily_performance (2025-01 sample): 1297 rows
fact_content_query_90d: 2414248 rows

=== DATE WINDOWS ===

Available months: 2025-01 to 2026-06
Total months: 18

First month (2025-01):
  Date range: 2025-01-27 to 2025-01-31
  Rows: 1297

Last month (2026-06):
  Date range: 2026-06-01 to 2026-06-30
  Rows: 11694072

=== MISSING VALUES (Features) ===

fact_content_daily_performance (2025-01 sample):
  gsc_impressions: 0 nulls (0.0%)
  gsc_clicks: 0 nulls (0.0%)
  ga4_pageviews: 0 nulls (0.0%)
  ga4_sessions: 0 nulls (0.0%)
  scroll_events: 0 nulls (0.0%)

dim_content:
  search_volume: 142622 nulls (27.4%)
  competition: 142622 nulls (27.4%)
  cpc: 142622 nulls (27.4%)
  backlinks: 267474 nulls (51.5%)
  keyword_char_count: 0 

What each query verifies:

Grain – confirms no duplicate (report_date, client_hash_id, content_hash_id) combinations
Row counts – prints the actual row counts for each table
Date windows – verifies coverage from 2025-01 through 2026-06, and sample dates within each month
Missing values – shows null percentages for key feature columns
Foreign keys – confirms all client_hash_id and content_hash_id in fact tables exist in dimension tables
Query 90d windows – verifies the fixed 90-day window boundaries
Value ranges – spot-checks that features have sensible min/max/mean values

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


What the data can never tell me:
- real query text, URLs, user identity, or internal ranking decisions
- a pre-made target label — labels must be derived from future windows
- anything outside the snapshot window or outside the pseudonymized IDs

Verified limits:
- history is unbalanced: the first partition only covers `2025-01-27` to `2025-01-31`, while later months extend through `2026-06-30`
- early rows are GSC-only: in the January 2025 sample, `gsc_data_available=True` and `ga4_data_available=False` for every row
- GA4 coverage is inconsistent later, so you cannot assume full GA4 history
- `fact_content_query_90d` is a 90-day aggregated window, not a daily fact table; it has a fixed window (`2026-04-02` to `2026-06-30`), so using it carelessly can cause future-data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.